In [1]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

In [2]:
import os
print(os.listdir("Images")[:10])

['n02097658-silky_terrier', 'n02092002-Scottish_deerhound', 'n02099849-Chesapeake_Bay_retriever', 'n02091244-Ibizan_hound', 'n02095314-wire-haired_fox_terrier', 'n02091831-Saluki', 'n02102318-cocker_spaniel', 'n02104365-schipperke', 'n02090622-borzoi', 'n02113023-Pembroke']


In [3]:
import torchvision
from torch.utils.data import random_split

full_dataset = torchvision.datasets.ImageFolder(root="Images")
class_names = [name.split("-", 1)[1].replace("_", " ") for name in full_dataset.classes]

print(f"{len(full_dataset)} total images, {len(class_names)} breeds")
print(class_names[:10])

20580 total images, 120 breeds
['Chihuahua', 'Japanese spaniel', 'Maltese dog', 'Pekinese', 'Shih-Tzu', 'Blenheim spaniel', 'papillon', 'toy terrier', 'Rhodesian ridgeback', 'Afghan hound']


In [4]:
#!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib

In [5]:
#!pip install -q "protobuf==3.20.3"

In [6]:
import transformers

In [11]:
#!pip show transformers
#!pip show protobuf
#!pip show tensorflow

In [8]:
#pip install --upgrade transformers datasets evaluate accelerate huggingface_hub

In [9]:
from transformers import ViTImageProcessor, ViTForImageClassification

# split
train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

# label mappings for the model config
id2label = {str(i): name for i, name in enumerate(class_names)}
label2id = {name: str(i) for i, name in enumerate(class_names)}

processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=len(class_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

Train: 17493, Val: 3087


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

[transformers] You passed `num_labels=120` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([120])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([120, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [12]:
import torch

def transform_example(example):
    image, label = example
    inputs = processor(images=image.convert("RGB"), return_tensors="pt")
    return {"pixel_values": inputs["pixel_values"][0], "labels": label}

class TransformedDataset(torch.utils.data.Dataset):
    def __init__(self, subset):
        self.subset = subset
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        return transform_example(self.subset[idx])

train_transformed = TransformedDataset(train_ds)
val_transformed = TransformedDataset(val_ds)

def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["labels"] for item in batch])
    return {"pixel_values": pixel_values, "labels": labels}

In [13]:
from torch.utils.data import Subset

# tiny slices just to confirm the training loop runs end-to-end
smoke_train = Subset(train_transformed, range(20))
smoke_val = Subset(val_transformed, range(10))

In [14]:
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="smoke_test_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="no",
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=smoke_train,
    eval_dataset=smoke_val,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

trainer.train()

/Users/melodyzi/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,4.925983,4.909513,0.000000


TrainOutput(global_step=5, training_loss=4.974269771575928, metrics={'train_runtime': 8.6188, 'train_samples_per_second': 2.321, 'train_steps_per_second': 0.58, 'total_flos': 1551478897704960.0, 'train_loss': 4.974269771575928, 'epoch': 1.0})